# Sport Elimination via Maximum Flow

This lecture explores the **Sport Elimination Problem**, a classic application of network flow algorithms. It demonstrates how to determine if a specific team can still mathematically finish at the top of a league table based on current standings and remaining matches.

---

## 1. Problem Setting and Context
The problem is modeled using a typical tournament format (e.g., the Indian Premier League) with the following assumptions:
* **League Matches only:** The focus is on regular-season standings, ignoring playoffs.
* **No Ties:** Every game results in a win for one team and a loss for the other.
* **Goal:** Determine if a "favorite team" can finish with at least as many wins as any other team (achieving at least a tie for the top spot).

### The Feasibility Question
The core task is to check if there exists *any* hypothetical scenario for the outcomes of the remaining games where the target team ends up on top. This is a question of mathematical possibility, not statistical probability.

---

## 2. Preliminary Analysis: The Greedy Assumption
To give the favorite team the best possible chance, we must assume a **Best-Case Scenario**:
* The favorite team wins **all** of its remaining matches.
* **Maximum Potential Score ($X$):** This is calculated as the sum of current wins and the number of remaining games for that team.

**Elimination Logic:**
If there is even one scenario where another team must inevitably end up with more than $X$ wins, the favorite team is mathematically eliminated.

---

## 3. Modeling with Network Flow
To solve the problem generally, we construct a flow network. This network helps distribute the "wins" from remaining matches among teams without any team exceeding the favorite team's maximum potential score.


### Network Components
* **Source ($S$) and Sink ($T$):** Designated start and end points for the flow.
* **Game Nodes:** A vertex for every pair of teams (excluding the favorite team) that still has matches to play against each other.
* **Team Nodes:** A vertex for every team in the league (excluding the favorite team).

### Edges and Capacities
The network is structured in layers:

1.  **Source to Game Nodes:**
    * **Edge:** From $S$ to each game node (e.g., Team A vs. Team B).
    * **Capacity:** The number of matches remaining between those two teams.
2.  **Game Nodes to Team Nodes:**
    * **Edge:** From a game node (Team A vs. Team B) to Team A and to Team B.
    * **Capacity:** Infinite (or the total number of remaining games), as the bottleneck is managed elsewhere.
3.  **Team Nodes to Sink:**
    * **Edge:** From each team node to $T$.
    * **Capacity:** $X - (\text{Current Wins of that team})$.
    * *Note:* This represents the "safety margin"—the maximum number of additional games the team can win without overtaking the favorite team.

---

## 4. Algorithmic Approach
The solution involves running a Max Flow algorithm (like Edmonds-Karp or Dinic's) on the constructed graph.

### Determining the Result
* **Successful (Not Eliminated):** If the **Maximum Flow** equals the **total number of matches** remaining between all other teams (i.e., all edges from the source are saturated).
    * This means all games can be assigned a winner such that no team exceeds the target score.
* **Eliminated:** If the Maximum Flow is **less** than the total number of remaining matches.
    * This indicates that in every possible scenario, at least one team will inevitably surpass the favorite team's score.

### Pseudocode for Sport Elimination
```python
FUNCTION CheckIfEliminated(FavoriteTeam, TeamsList, MatchesRemaining, CurrentWins):
    1. Calculate MaxPotentialWins = CurrentWins[FavoriteTeam] + MatchesRemaining[FavoriteTeam]
    
    2. Initialize FlowNetwork:
        - Add Source S and Sink T
        - FOR each pair of teams (A, B) where A, B != FavoriteTeam:
            - Create a GameNode(A, B)
            - Add Edge(S, GameNode) with Capacity = MatchesRemainingBetween(A, B)
            - Add Edge(GameNode, TeamNode A) with Capacity = Infinity
            - Add Edge(GameNode, TeamNode B) with Capacity = Infinity
            
        - FOR each Team T in TeamsList (where T != FavoriteTeam):
            - SafetyMargin = MaxPotentialWins - CurrentWins[T]
            - IF SafetyMargin < 0:
                RETURN "Eliminated" // Already impossible
            - Add Edge(TeamNode T, Sink T) with Capacity = SafetyMargin
            
    3. Calculate MaxFlow from S to T in the FlowNetwork
    
    4. Calculate TotalRemainingGamesBetweenOthers = Sum of all MatchesRemainingBetween(A, B)
    
    5. IF MaxFlow == TotalRemainingGamesBetweenOthers:
        RETURN "Not Eliminated"
    ELSE:
        RETURN "Eliminated"
```

---

## 5. Summary of Takeaways
* **Modeling Versatility:** The Sport Elimination problem highlights how network flows can solve complex combinatorial problems that do not initially look like graph problems.
* **Flow as Distribution:** In this model, "flow" represents "wins." The source provides the total wins available from games, and the sink limits how many wins each team can "absorb".
* **Integral Flow:** Because match wins must be whole numbers, the **Integrality Theorem** of Max Flow is crucial; it ensures that if capacities are integers, the flow values (match outcomes) will also be integers .
* **Future Concept (Duality):** When a team is eliminated, the "Min-Cut" in the graph provides a succinct explanation (evidence) of why no scenario allows that team to win, which is more efficient than checking all possible match outcomes 